In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'mediapipe==0.10.5', '--no-deps', '--force-reinstall',
                '--quiet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'tensorflow', 'scikit-learn', 'matplotlib',
                '--quiet'], check=True)

In [ ]:
import cv2
import itertools
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import os
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print(f'mediapipe  {mp.__version__}')
print(f'opencv     {cv2.__version__}')
print(f'tensorflow {tf.__version__}')

## Etapa 1 — Coleta

In [ ]:
SINAIS               = ['Oi', 'Tudo_bem', 'Tchau']
NUM_AMOSTRAS         = 30
FRAMES_POR_AMOSTRA   = 30
PASTA_DADOS          = 'dados_libras'
COUNTDOWN            = 3
PAUSA_ENTRE_AMOSTRAS = 1.5

for sinal in SINAIS:
    os.makedirs(os.path.join(PASTA_DADOS, sinal), exist_ok=True)

In [ ]:
def extrair_landmarks(results):
    pose = (
        np.array([[lm.x, lm.y, lm.z, lm.visibility]
                  for lm in results.pose_landmarks.landmark]).flatten()
        if results.pose_landmarks else np.zeros(33 * 4)
    )
    mao_esq = (
        np.array([[lm.x, lm.y, lm.z]
                  for lm in results.left_hand_landmarks.landmark]).flatten()
        if results.left_hand_landmarks else np.zeros(21 * 3)
    )
    mao_dir = (
        np.array([[lm.x, lm.y, lm.z]
                  for lm in results.right_hand_landmarks.landmark]).flatten()
        if results.right_hand_landmarks else np.zeros(21 * 3)
    )
    return np.concatenate([pose, mao_esq, mao_dir])  # (258,)

In [ ]:
mp_holistic = mp.solutions.holistic

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError('nao foi possivel abrir a camera')

with mp_holistic.Holistic(min_detection_confidence=0.5,
                           min_tracking_confidence=0.5) as holistic:
    for sinal in SINAIS:
        print(f'\n--- {sinal} ---')
        input('Enter quando pronto...')

        for i in range(NUM_AMOSTRAS):
            for t in range(COUNTDOWN, 0, -1):
                ret, frame = cap.read()
                if ret:
                    cv2.putText(frame, str(t), (30, 80),
                                cv2.FONT_HERSHEY_SIMPLEX, 2.5, (0, 255, 255), 3)
                    cv2.imshow('coleta', frame)
                    cv2.waitKey(1000)

            sequencia = []
            for j in range(FRAMES_POR_AMOSTRA):
                ret, frame = cap.read()
                if not ret:
                    break

                rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(rgb)
                sequencia.append(extrair_landmarks(results))

                cv2.putText(frame, f'{sinal}  {i+1}/{NUM_AMOSTRAS}  f{j+1}',
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.imshow('coleta', frame)

                if cv2.waitKey(1) & 0xFF == ord('q'):
                    raise KeyboardInterrupt

            np.save(os.path.join(PASTA_DADOS, sinal, f'amostra_{i:03d}.npy'),
                    np.array(sequencia))
            print(f'  [{i+1}/{NUM_AMOSTRAS}] salvo')
            time.sleep(PAUSA_ENTRE_AMOSTRAS)

cap.release()
cv2.destroyAllWindows()
print('concluido')

In [ ]:
print(f'{"sinal":<14} {"amostras":>9} {"shape":>12} {"zeros%":>8}')
print('-' * 48)

for sinal in SINAIS:
    pasta    = os.path.join(PASTA_DADOS, sinal)
    arquivos = sorted(f for f in os.listdir(pasta) if f.endswith('.npy'))
    ok, zeros = 0, []

    for arq in arquivos:
        d = np.load(os.path.join(pasta, arq))
        if d.shape == (FRAMES_POR_AMOSTRA, 258):
            ok += 1
        zeros.append((d == 0).mean() * 100)

    pz  = np.mean(zeros) if zeros else 0
    shp = f'({FRAMES_POR_AMOSTRA}, 258)' if ok == len(arquivos) else 'SHAPE ERRADO'
    print(f'{sinal:<14} {len(arquivos):>9} {shp:>12} {pz:>7.1f}%')

    if pz > 40:
        print(f'  aviso: muitos zeros em {sinal} — mediapipe nao detectou bem')

## Etapa 2 — Dataset

In [ ]:
DIM_LANDMARKS = 258
label_map     = {sinal: i for i, sinal in enumerate(SINAIS)}

X, y, problemas = [], [], []

for sinal in SINAIS:
    pasta    = os.path.join(PASTA_DADOS, sinal)
    arquivos = sorted(f for f in os.listdir(pasta) if f.endswith('.npy'))

    for arq in arquivos:
        dados = np.load(os.path.join(pasta, arq))
        if dados.shape != (FRAMES_POR_AMOSTRA, DIM_LANDMARKS):
            problemas.append(f'{sinal}/{arq}: shape errado {dados.shape}')
            continue
        X.append(dados)
        y.append(label_map[sinal])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)

print(f'X: {X.shape}')
print(f'y: {y.shape}')

if problemas:
    for p in problemas:
        print(f'  ignorado: {p}')

In [ ]:
print('amostras por sinal:')
for sinal, idx in label_map.items():
    n = (y == idx).sum()
    print(f'  {sinal:<14} {n:3d}  {"█" * n}')

pct_zeros = (X == 0).mean() * 100
print(f'\nzeros no dataset: {pct_zeros:.1f}%')
if pct_zeros > 40:
    print('aviso: muitos zeros — considere regravar com melhor iluminacao')

In [ ]:
fig, axes = plt.subplots(1, len(SINAIS), figsize=(14, 3), sharey=False)

for ax, (sinal, idx) in zip(axes, label_map.items()):
    std_por_frame = X[y == idx].std(axis=0).mean(axis=1)
    ax.plot(std_por_frame, color='steelblue', linewidth=1.5)
    ax.fill_between(range(30), std_por_frame, alpha=0.2, color='steelblue')
    ax.set_title(sinal.replace('_', ' '), fontsize=11)
    ax.set_xlabel('frame')
    ax.set_ylabel('desvio padrao medio')
    ax.grid(alpha=0.3)

plt.suptitle('variacao dos landmarks por frame (maior = mais movimento)', fontsize=11)
plt.tight_layout()
plt.savefig('variacao_landmarks.png', dpi=120)
plt.show()

In [ ]:
np.save('X.npy', X)
np.save('y.npy', y)

print(f'X.npy salvo  {X.nbytes / 1e6:.1f} MB')
print(f'y.npy salvo  {y.nbytes / 1e3:.1f} KB')

## Etapa 3 — Treinamento

In [ ]:
y_cat = keras.utils.to_categorical(y, num_classes=len(SINAIS))

X_train, X_val, y_train, y_val = train_test_split(
    X, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'treino:    {X_train.shape}')
print(f'validacao: {X_val.shape}')

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(30, 258)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(SINAIS), activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='treino')
axes[0].plot(history.history['val_loss'], label='validacao')
axes[0].set_title('loss')
axes[0].set_xlabel('epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='treino')
axes[1].plot(history.history['val_accuracy'], label='validacao')
axes[1].set_title('acuracia')
axes[1].set_xlabel('epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_treino.png', dpi=120)
plt.show()

loss, acc = model.evaluate(X_val, y_val, verbose=0)
print(f'acuracia na validacao: {acc*100:.1f}%')

In [ ]:
y_pred  = model.predict(X_val, verbose=0)
y_true  = np.argmax(y_val,  axis=1)
y_pred_ = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_true, y_pred_)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ticks = range(len(SINAIS))
ax.set_xticks(list(ticks))
ax.set_yticks(list(ticks))
ax.set_xticklabels([s.replace('_', ' ') for s in SINAIS])
ax.set_yticklabels([s.replace('_', ' ') for s in SINAIS])
ax.set_xlabel('predito')
ax.set_ylabel('real')
ax.set_title('matriz de confusao')

for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, str(cm[i, j]), ha='center', va='center',
            color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=120)
plt.show()

print(classification_report(y_true, y_pred_,
                             target_names=[s.replace('_', ' ') for s in SINAIS]))

## Exportação — TF.js

In [ ]:
model.save('libras_model.keras')
print('libras_model.keras salvo')

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, 'exportar_tfjs.py',
                '--modelo', 'libras_model.keras',
                '--saida', 'libras_model_tfjs'], check=True)
print('convertido - pasta libras_model_tfjs/ criada')

In [ ]:
import tensorflowjs as tfjs

model = tf.keras.models.load_model('libras_model.keras')

tfjs.converters.save_keras_model(model, 'libras_model_tfjs')